Ensure the library chromadb is latest. Run the below command

## imports

In [565]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
import chromadb
import os

from langchain_chroma import Chroma
from dotenv import load_dotenv  
load_dotenv()

# from google.colab import userdata



True

Below is the api key, endpoint, api version, deployment name for the chat model

In [566]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [567]:
from groq import Groq

client = Groq()

In [568]:
model_name = 'llama-3.1-8b-instant'

 Instantiating the embedding model

In [155]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Load the Vector Database

In [6]:
import zipfile

with zipfile.ZipFile("tesla-annual-reports.zip", "r") as zip_ref:
    zip_ref.extractall("tesla-annual-reports")
print("Extraction completed.")

Extraction completed.


## chunking 


In [7]:
from pathlib import Path
pdf_folder_location = Path("tesla-annual-reports/tesla-annual-reports").resolve()
print("Loading PDFs from:", pdf_folder_location)

Loading PDFs from: C:\Users\Aman Choudhary\Desktop\git_task\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\Amanchoudhary\tesla-annual-reports\tesla-annual-reports


In [569]:
pdf_loader = PyPDFDirectoryLoader(str(pdf_folder_location))

In [570]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [571]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [65]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)
len(tesla_10k_chunks)

3337

In [572]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

# Database Creation

In [573]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)
 

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [574]:
chromadb_client.heartbeat()

1780548463643835500

In [575]:
chromadb_client

In [576]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [577]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [578]:
chromadb_client.count_collections()

1

In [19]:
i = 0 
 
while i < len(tesla_10k_chunks):
    vectorstore.add_documents( 
        documents=tesla_10k_chunks[i:i+500], 
        ids=["text_" + str(i) for i in range(i, i+500)] 
    )
 
    i += 500 
    time.sleep(5)

## Retrieving relevant records

In [579]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [580]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [581]:
user_query = "Automotive revenue in 2021?"

In [582]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000232439CD610>, search_kwargs={'k': 5})

In [583]:
retriever.invoke(user_query)

[Document(id='text_3106', metadata={'creationdate': '2024-01-29T11:11:14+00:00', 'creator': 'wkhtmltopdf 0.12.6', 'page': 38, 'page_label': '39', 'producer': 'Qt 5.15.2', 'source': 'C:\\Users\\Aman Choudhary\\Desktop\\git_task\\AI_Training_Batch_May_2026\\submissions\\assignments\\assignment-02\\Amanchoudhary\\tesla-annual-reports\\tesla-annual-reports\\tsla-20231231-gen.pdf', 'title': '', 'total_pages': 130}, page_content='2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13

In [584]:
len(retriever.invoke(user_query))

5

## Basic Rag

In [758]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [759]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [587]:
# questions

# Q1

In [763]:
user_query = "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A."

# Q2

In [795]:
#Irrelevent Query
user_query="Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims. "

# Q3

In [826]:
user_query = "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations. aa"

# Q4

In [857]:
user_query = "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"

In [858]:
relevant_document_chunks = retriever.invoke(user_query)

In [859]:
model_name = 'openai/gpt-oss-120b'

docs = retriever.invoke(user_query)
#docs = vectorstore_persisted.similarity_search_with_score(user_query, k=5)
context_list = [doc.page_content for doc in docs]
#context_list = [d.page_content for d in docs]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

**Strategic importance**

- **Automotive operations** are the dominant engine of the business. The segment generates the bulk of revenue (e.g., $29.5 billion in 2020 versus $2.0 billion for energy) and delivers the largest disclosed gross profit ($6.6 billion in 2020). It also encompasses the full value chain—design, development, manufacturing, sales, leasing, regulatory‑credit sales, after‑sales services, used‑vehicle sales, merchandise and insurance—making it the primary source of cash flow and earnings.

- **Energy generation and storage** is a smaller but strategically complementary segment. It supports the company’s broader mission of accelerating the transition to sustainable energy and provides diversification beyond vehicle sales. The segment includes solar‑energy generation, energy‑storage products, related services and incentive sales, positioning the firm to capture growth in renewable‑energy markets and to leverage emerging AI, robotics and automation capabilities.

**Discl

In [860]:
baseline_top_chunks = []

for idx, doc in enumerate(docs):

    baseline_top_chunks.append({
        "chunk_id": f"chunk_{idx+1}",
        "section": "Unknown",
        "year": 2023
    })

# RAG With Query Expansion Technique

In [861]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [862]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [863]:
model_name='openai/gpt-oss-120b'

In [864]:
prompt=[
    {'role':'system', 'content':query_expansion_system_message},
    {'role':'user',   'content': user_message_template.format(
        question=user_query
    )}
]

In [865]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nCompare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?\n</Question>\n'}]

In [866]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [867]:
type(query_expansions.choices[0].message.content)

str

In [868]:
print(query_expansions.choices[0].message.content)

How does the 10‑K illustrate the strategic importance of automotive operations versus energy generation/storage, and what disclosures are required to back a business recommendation?  
Compare the strategic significance of the automotive segment and the energy generation/storage segment using evidence from the 10‑K, and identify which disclosures are needed to support a business recommendation.  
What does the 10‑K reveal about the relative strategic value of automotive operations and energy generation/storage, and what specific disclosures must be included to justify a business recommendation?


In [869]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [870]:
print(query_expansions)

ChatCompletion(id='chatcmpl-5ec5ad6f-40c8-4eba-bb07-bbb1f1036543', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='How do the automotive operations and energy generation/storage segments differ in strategic importance according to the 10‑K, and what specific disclosures are required to justify a business recommendation?  \nWhat evidence from the 10‑K highlights the strategic significance of the automotive versus the energy generation/storage businesses, and which disclosures must be included to support a recommendation?  \nIn comparing the strategic value of automotive operations with energy generation and storage in the 10‑K, what disclosures are necessary to underpin a business recommendation?  \nBased on the 10‑K, how important are the automotive and energy generation/storage segments strategically, and what key disclosures should be referenced to back a business recommendation?', role='assistant', annotations=None, executed_tools

In [871]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets, no extra text. Need to rephrase the question in multiple ways, preserving meaning. Provide at least 3 versions.

Original: "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"

We can produce variations: "How do automotive operations and energy generation/storage compare in strategic importance according to the 10-K, and which disclosures are required to back a business recommendation?" etc.

Make sure each line is a question. Provide at least 3. No bullet points. No numbering. Just list.

Let's produce 4 variations.


In [872]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [873]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce at least 3 versions of the question, each on a new line, no numbering or bullets, no extra text. Need to rephrase the question in multiple ways, preserving meaning. Provide at least 3 versions.

Original: "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"

We can produce variations: "How do automotive operations and energy generation/storage compare in strategic importance according to the 10-K, and which disclosures are required to back a business recommendation?" etc.

Make sure each line is a question. Provide at least 3. No bullet points. No numbering. Just list.

Let's produce 4 variations.


In [874]:
len(query_expansions_list)

4

In [875]:
len(query_expansions_list)

4

In [876]:
query_expansions_list

['How do the automotive operations and energy generation/storage segments differ in strategic importance according to the 10‑K, and what specific disclosures are required to justify a business recommendation?  ',
 'What evidence from the 10‑K highlights the strategic significance of the automotive versus the energy generation/storage businesses, and which disclosures must be included to support a recommendation?  ',
 'In comparing the strategic value of automotive operations with energy generation and storage in the 10‑K, what disclosures are necessary to underpin a business recommendation?  ',
 'Based on the 10‑K, how important are the automotive and energy generation/storage segments strategically, and what key disclosures should be referenced to back a business recommendation?']

In [877]:
expanded_context_list = []

In [878]:
expanded_context_list = []
expanded_top_chunks = []

for query in query_expansions_list:

    # docs = vectorstore_persisted.similarity_search_with_score(
    #     query,
    #     k=3
    # )
        docs = retriever.invoke(query)
        for doc  in docs:
            expanded_context_list.append(
            doc.page_content
        )

        expanded_top_chunks.append({
            "chunk_id": f"page_{doc.metadata['page']}",
            "section": "Unknown",
            "year": 2023,
            "retrieved_by": query
        })

In [879]:
citations = []

seen = set()

for chunk in expanded_top_chunks:

    if chunk["chunk_id"] not in seen:

        seen.add(chunk["chunk_id"])

        citations.append({
            "chunk_id": chunk["chunk_id"],
            "source_doc": "tsla-20231231-gen.pdf",
            "section": chunk["section"],
            "year": chunk["year"]
        })

In [880]:
len(expanded_context_list)

12

In [881]:
expanded_context_list

['Note\t\n2\n1\n\t–\tSegment\tReporting\tand\tInformation\tabout\tGeographic\tAreas\nWe\thave\t\ntwo\n\toperating\tand\treportable\tsegments:\t(i)\tautomotive\tand\t(ii)\tenergy\tgeneration\tand\tstorage.\tThe\tautomotive\tsegment\tincludes\tthe\tdesign,\ndevelopment,\tmanufacturing,\tsales,\tand\tleasing\tof\telectric\tvehicles\tas\twell\tas\tsales\tof\tautomotive\tregulatory\tcredits.\tAdditionally,\tthe\tautomotive\tsegment\tis\nalso\tcomprised\tof\tservices\tand\tother,\twhich\tincludes\tnon-warranty\tafter-sales\tvehicle\tservices,\tsales\tof\tused\tvehicles,\tretail\tmerchandise,\tsales\tby\tour\tacquired\nsubsidiaries\tto\tthird\tparty\tcustomers,\tand\tvehicle\tinsurance\trevenue.\tThe\tenergy\tgeneration\tand\tstorage\tsegment\tincludes\tthe\tdesign,\tmanufacture,\ninstallation,\tsales,\tand\tleasing\tof\tsolar\tenergy\tgeneration\tand\tenergy\tstorage\tproducts\tand\trelated\tservices\tand\tsales\tof\tsolar\tenergy\tsystems\tincentives.\tOur\nCODM\tdoes\tnot\tevaluate\toperat

In [882]:
final_context_documents = set(expanded_context_list)

In [883]:
len(final_context_documents) 

6

In [884]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [885]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [886]:
model_name = 'openai/gpt-oss-120b'

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=final_context_documents,
         question=user_query
        )
    }
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)


**Strategic importance**

- **Automotive operations** are the company’s primary revenue engine.  In 2020 the automotive segment generated **$29.5 billion in revenue** and **$6.6 billion of gross profit**, far out‑pacing the energy segment.  The segment covers the full value chain—design, development, manufacturing, sales, leasing, and the sale of automotive regulatory credits—as well as a broad suite of services (used‑vehicle sales, non‑warranty after‑sales service, parts, retail merchandise, paid Supercharging, vehicle‑insurance revenue, etc.  This breadth makes automotive the core driver of cash flow and profitability and the main lever for scaling the company’s mission to accelerate sustainable transportation.

- **Energy generation and storage** is a smaller but strategically complementary business.  It contributes **$2.0 billion in revenue** (2020) and focuses on the design, manufacture, installation, sales and leasing of solar‑energy generation systems, energy‑storage products, a

In [887]:
import json
import os

json_file = "query_expansion_results.json"

if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        data = [data]
else:
    data = []

question_id = f"Q{len(data) + 1}"

output_schema = {
    "question_id": question_id,
    "original_query": user_query,
    "expanded_queries": query_expansions_list,
    "baseline_top_chunks": baseline_top_chunks,
    "expanded_top_chunks": expanded_top_chunks,
    "final_answer": prediction,
    "citations": citations,
    "retrieval_improvement_analysis":
        "Query expansion improved retrieval coverage compared to baseline retrieval."
}

data.append(output_schema)

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)